# Génération de CRH fictifs en CIM-11

Ce notebook est une adaptation de `generate_scenarios_v4.ipynb` (repo `recode-scenario`) pour générer des comptes rendus hospitaliers fictifs annotés en **CIM-11** au lieu de CIM-10.

## Pipeline général
1. Lecture des scénarios PMSI (codes CIM-10) depuis le fichier Parquet
2. Transcodage CIM-10 → CIM-11 via la table de mapping OMS
3. Pour chaque code CIM-11, tirage au sort de codes post-coordonnés avec vérification de cohérence via Mistral
4. Pour chaque DAS, tirage au sort d'un code post-coordonné avec vérification de cohérence DAS/DP et entre DAS
5. Construction des prompts et génération des CRH via l'API Mistral
6. Sauvegarde du fichier final — **une ligne par CRH**

## Modifications apportées par rapport à l'original
1. **Transcodage CIM-10 → CIM-11** via la table de mapping OMS (`10To11MapToOneCategory.xlsx`)
2. **Postcoordination DP** : tirage au sort de 5 codes post-coordonnés avec vérification de cohérence patient/code via Mistral
3. **Postcoordination DAS** : tirage au sort d'1 code post-coordonné par DAS avec vérification cohérence DAS/DP puis entre tous les DAS
4. **Templates** : remplacement de CIM-10 par CIM-11 + instruction pour décrire chaque élément de la postcoordination
5. **Diagnostics secondaires** : filtrés par cohérence, sauvegardés dans `icd11_secondary_code` (filtrés) et `icd11_secondary_code_all` (tous)
6. **Structure de sortie** : une ligne par CRH (chaque séjour peut générer plusieurs CRH selon les codes post-coordonnés disponibles)
7. **Dossier horodaté** : chaque exécution crée un dossier unique pour les résultats

## Fichiers nécessaires
- `Data>SCENARIOS>scenarios_bn_all_20260128.pq` — scénarios PMSI (nouveau fichier)
- `referentials/` — référentiels médicaux (CIM-10, CCAM, cancer, spécialités)
- `Dictionnaire/CIM11/mapping/10To11MapToOneCategory.xlsx` — table de mapping OMS
- `api/data/cim11_termes.csv` — dictionnaire CIM-11 avec libellés français
- `api/code_post-coordonees/cim11_postcoord.csv` — codes post-coordonnés générés
- `config.py` — clé API Mistral et chemin des résultats
- `utils_v2.py` — fonctions utilitaires du pipeline original

## 1. Paramètres de configuration
Définition des paramètres globaux du pipeline.
- `N_SCENARIO` : nombre de séjours à traiter
- `QUERY` : filtre optionnel sur les scénarios (ex: exclure les cancers)

In [1]:
profile_file = "scenarios_bn_all_20260128.pq"

# Nombre de scénarios à générer
N_SCENARIO = 200

# Filtre optionnel sur les scénarios
# Exemple pour exclure les cancers : QUERY = "icd_primary_code.isin(@gs.icd_codes_cancer)"
QUERY = None

## 2. Initialisation de l'environnement
Montage du Drive Google et ajout du chemin vers les fichiers du projet dans `sys.path`.

> **MODIF CIM-11** : Cellule ajoutée — nécessaire pour accéder aux fichiers sur Google Drive depuis Colab et pour que Python trouve `utils_v2.py`.

In [2]:
# MODIF CIM-11 : Montage du Drive et ajout du chemin vers utils_v2.py
# Sans sys.path.append, Python ne trouve pas utils_v2.py qui est sur le Drive
import sys
from google.colab import drive
drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh')

Mounted at /content/drive


## 3. Installation des dépendances
> **MODIF CIM-11** : Cellule ajoutée.
>
> **Problème** : `utils_v2.py` utilise `from mistralai import File, Mistral`. La version par défaut de Colab ne contient pas la classe `File`.
>
> **Solution** : forcer l'installation de la version 1.2.0 compatible.
>
> Les avertissements de compatibilité affichés sont non bloquants.

In [3]:
# MODIF CIM-11 : Installation de mistralai 1.2.0 compatible avec utils_v2.py
# La version par défaut de Colab ne contient pas la classe File nécessaire dans utils_v2.py
# Si cette commande échoue (package en quarantaine), utiliser :
# !pip install git+https://github.com/mistralai/client-python.git -q
!pip install mistralai==1.2.0 -q --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

## 4. Imports

In [4]:
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm
import os
import json
import random
import math
from utils_v2 import *

## 5. Configuration et initialisation
Chargement de la config (clé API, chemins) et définition des années de simulation.

In [5]:
# Chargement de la clé API Mistral et du chemin des résultats depuis config.py
# config.py contient : api_key et path_results
from config import *
simulations_years = [dt.date.today().year-2, dt.date.today().year-1, dt.date.today().year]

## 6. Initialisation de l'objet de génération
> **MODIF CIM-11** : Cellule modifiée.
>
> **Problème** : `utils_v2.py` utilise des chemins relatifs codés en dur (`templates/`, `referentials/`). Par défaut, Colab travaille depuis `/content/` et ne trouve pas ces dossiers.
>
> **Solution** : `os.chdir()` change le répertoire de travail vers `generation_crh/` pour que les chemins relatifs fonctionnent.

In [6]:
# MODIF CIM-11 : Changement du répertoire de travail
# utils_v2.py cherche les fichiers avec des chemins relatifs comme "templates/regles_atih.yml"
# On doit donc travailler depuis le dossier generation_crh/ pour que ces chemins fonctionnent
os.chdir('/content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh')

PATH_REF = 'referentials/'
PATH_DATA = 'data/'

gs = generate_scenario(path_ref=PATH_REF, path_data=PATH_DATA)
gs.simulations_years = simulations_years

## 7. Chargement des référentiels médicaux
Chargement des dictionnaires CIM-10, actes CCAM, recommandations cancer, spécialités et hôpitaux.
Ces référentiels sont utilisés par `generate_scenario_from_profile()` pour enrichir les scénarios.

In [7]:
gs.load_offical_icd("CIM_ATIH_2025/LIBCIM10MULTI.TXT", col_names=["icd_code","aut_mco","pos","aut_ssr","icd_code_description_short","icd_code_description"])
gs.load_icd_categ_weight("ponderation_code_categ.csv", col_names={"diag":"icd_code","ponderation":"weight"})
gs.load_offical_procedures("ccam_actes_2024.xlsx", col_names={"code":"procedure","libelle_long":"procedure_description"})
col_names={"Code CIM":"icd_parent_code","Localisation":"primary_site","Type Histologique":"histological_type",
           "Stade":"stage","Marqueurs Tumoraux":"biomarkers","Traitement":"treatment_recommandation","Protocole de Chimiothérapie":"chemotherapy_regimen"}
gs.load_cancer_treatement_recommandations("Tableau récapitulatif traitement cancer.xlsx", col_names)
col_names={"racine":"drg_parent_code","lib_spe_uma":"specialty","ratio_spe_racine":"ratio"}
gs.load_specialty_refential("dictionnaire_spe_racine.xlsx", col_names)
gs.load_referential_hospital("chu")
gs.load_exclusions("exclusions")

/content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/utils_v2.py:315: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_icd.code = df_icd.icd_code.str.replace(" ","")


## 8. Chargement du profil de classification
Chargement du fichier Parquet des scénarios PMSI avec renommage des colonnes.

**Note** : Le fichier doit être dans `data/` **sans extension** `.pq` car `utils_v2.py` cherche `data/` + `file_name` sans extension.

> **MODIF nouveau fichier** : Dans `scenarios_bn_all_20260128.pq`, la colonne `age` contient des tranches d'âge textuelles (`ge_18`, `lt_18`) et non l'âge numérique. C'est la colonne `agean` qui contient l'âge numérique → on la mappe vers `age2`. La colonne `duree` est renommée en `los`.

###  Test — Vérification des colonnes d'âge du nouveau fichier
Vérifie que `agean` contient bien l'âge numérique et que `age` contient des tranches textuelles. Cela justifie le choix de mapper `agean → age2` dans `col_names`.

In [8]:
import pyarrow.parquet as pq
table = pq.read_table('/content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/data/scenarios_bn_all_20260128.pq')
df_test = table.to_pandas()
print(df_test[['age', 'agean']].head(10))
print(f"\nType age   : {df_test['age'].dtype}")
print(f"Type agean : {df_test['agean'].dtype}")

     age  agean
0  ge_18   77.0
1  ge_18   72.0
2  ge_18   44.0
3  ge_18   33.0
4  ge_18   73.0
5  ge_18   28.0
6  ge_18   52.0
7  ge_18   27.0
8  ge_18   43.0
9  ge_18   35.0

Type age   : object
Type agean : float64


In [9]:
# Renommage des colonnes PMSI en noms plus explicites
# diag2 → icd_primary_code (diagnostic principal CIM-10)
# diagnostic_associes → icd_secondary_code (diagnostics secondaires CIM-10)
# MODIF : agean → age2 (âge numérique, la colonne 'age' contient des tranches d'âge textuelles)
# MODIF : duree → los (durée de séjour)
col_names={"racine":"drg_parent_code","diagnostic_associes": "icd_secondary_code","diag2":"icd_primary_code",
            "mdp":"case_management_type","n":"nb",
            "mode_entree":"admission_mode",
            "mode_sortie":"discharge_disposition",
            "mode_hospit":"admission_type",
            "duree":"los",
            "agean":"age2",
            "nbda":"nb_associated"}

gs.load_classification_profile(profile_file, col_names, replace=True)

### Test — Vérification de la colonne los
Vérifie que la colonne `los` (durée de séjour) est bien présente et contient des valeurs cohérentes.

In [10]:
# MODIF : suppression des lignes avec los NaN
gs.df_classification_profile = gs.df_classification_profile[
    gs.df_classification_profile['los'].notna()
].reset_index(drop=True)
print(f"Scénarios après suppression los NaN : {len(gs.df_classification_profile)}")

# Test : vérification de la colonne los
gs.df_classification_profile.loc[gs.df_classification_profile.los>2, ["drg_parent_code","icd_primary_code","icd_secondary_code","los"]]

Scénarios après suppression los NaN : 149014


,drg_parent_code,icd_primary_code,icd_secondary_code,los
1,07K06,C220,"[K703, K746, B181, E1198, F10202, G473, K758, ...",3.0
3,27C06,N185,"[Z290, Z713, I770, Z292, Z4580, Z488]",20.0
4,01M08,G20,"[I951, R471, E785, R410, Z4584]",27.0
5,14Z13,O800,"[O431, Z370, D259, J450, O234, O702, O720, Z59...",5.0
7,14Z14,O800,"[Z300, Z352, Z370, Z3908, Z391]",4.0
...,...,...,...,...
149008,11M04,N390,"[B962, R410, E559, I482, R600, Z741]",47.0
149009,11M04,N390,"[B962, R410, E559, I482, R600, Z741]",47.0
149010,15M05,Z380,"[P002, P033, P201, P211, Z1351]",4.0
149011,15M06,Z380,"[Q663, Z1351]",5.0


## 9. Chargement des diagnostics secondaires et procédures
Chargement des diagnostics secondaires PMSI et des actes CCAM associés à chaque profil.
Ces données permettent d'enrichir les scénarios avec des comorbidités et actes médicaux.

In [11]:
col_names={"racine":"drg_parent_code","das": "icd_secondary_code","diag":"icd_primary_code","categ_cim":"icd_primary_parent_code",
            "mdp":"case_management_type","nb_situations":"nb","acte":"procedure",
            "mode_entree":"admission_mode",
            "mode_sortie":"discharge_disposition",
            "mode_hospit":"admission_type"}

gs.load_secondary_icd("bn_pmsi_related_diag_20250818.csv", col_names)
gs.load_procedures("bn_pmsi_procedures_20250818.csv", col_names)

## 10. Transcodage CIM-10 → CIM-11
Chargement de la table de mapping OMS et création des fonctions de normalisation et de transcodage.

> **MODIF CIM-11** : Cellule ajoutée.
>
> **Problème** : Les scénarios PMSI utilisent des codes CIM-10. On doit les convertir en CIM-11.
>
> **Solution** : Table de mapping officielle de l'OMS + troncature progressive.
>
> **Troncature progressive** : Si le code exact n'existe pas dans la table (ex: `S72.00`), on essaie une version moins précise (`S72.0` puis `S72`). Cela permet de passer de 86% à 99.4% de conversion.

In [12]:
# MODIF CIM-11 : Chargement de la table de mapping officielle OMS CIM-10 → CIM-11
df_mapping = pd.read_excel(
    '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/Dictionnaire/CIM11/mapping/10To11MapToOneCategory.xlsx'
)

# On garde uniquement les lignes où les deux codes sont présents (pas de valeurs vides)
df_mapping_clean = df_mapping[
    df_mapping['icd10Code'].notna() & df_mapping['icd11Code'].notna()
][['icd10Code', 'icd11Code']].copy()

# MODIF CIM-11 : Normalisation du format CIM-10
# Le fichier Parquet stocke les codes sans point (ex: C180)
# La table de mapping les a avec point (ex: C18.0)
# On insère donc le point après le 3ème caractère si le code a 4+ caractères
def normaliser_code_cim10(code):
    code = str(code).strip()
    if '.' in code:      # Déjà normalisé
        return code
    if len(code) >= 4:   # Ex: C180 → C18.0
        return code[:3] + '.' + code[3:]
    return code          # Code court (ex: M23) → on le laisse tel quel

df_mapping_clean['icd10Code_norm'] = df_mapping_clean['icd10Code'].apply(normaliser_code_cim10)

# Construction du dictionnaire de mapping {code_cim10_normalisé: code_cim11}
dict_mapping = dict(zip(df_mapping_clean['icd10Code_norm'], df_mapping_clean['icd11Code'].str.strip()))

# MODIF CIM-11 : Transcodage avec troncature progressive
# Essai 1 : code exact (ex: S72.00)
# Essai 2 : code tronqué d'un chiffre (ex: S72.0)
# Essai 3 : code sans décimale (ex: S72)
# → permet de passer de 86% à 99.4% de conversion
def transcoder_avec_troncature(code_cim10):
    norm = normaliser_code_cim10(code_cim10)
    if norm in dict_mapping:
        return dict_mapping[norm]
    if '.' in norm:
        tronque = norm[:-1]
        if tronque in dict_mapping:
            return dict_mapping[tronque]
        sans_decimal = norm.split('.')[0]
        if sans_decimal in dict_mapping:
            return dict_mapping[sans_decimal]
    return None  # Aucune correspondance trouvée

# Test de la fonction de transcodage
print(f"Codes uniques CIM-10 dans la table : {df_mapping_clean['icd10Code_norm'].nunique()}")
print(f"Exemple code exact    : C18.0 → {transcoder_avec_troncature('C180')}")
print(f"Exemple avec troncature : S72.00 → {transcoder_avec_troncature('S7200')}")

Codes uniques CIM-10 dans la table : 12499
Exemple code exact    : C18.0 → 2B90.0Z&XA6J68
Exemple avec troncature : S72.00 → NC72.2Z


### Test — Fonctions de normalisation et de transcodage
Vérifie que les fonctions de normalisation et de transcodage fonctionnent correctement sur quelques exemples.

In [13]:
# Test de normaliser_code_cim10
print("=== Test normalisation ===")
print(f"C180  → {normaliser_code_cim10('C180')}")   # Attendu: C18.0
print(f"S7200 → {normaliser_code_cim10('S7200')}")  # Attendu: S72.00
print(f"M23   → {normaliser_code_cim10('M23')}")    # Attendu: M23 (code court)
print(f"C18.0 → {normaliser_code_cim10('C18.0')}")  # Déjà normalisé

# Test de transcoder_avec_troncature
print("\n=== Test transcodage ===")
print(f"C180  (code exact)      → {transcoder_avec_troncature('C180')}")
print(f"S7200 (troncature 1)    → {transcoder_avec_troncature('S7200')}")
print(f"S72   (sans décimale)   → {transcoder_avec_troncature('S72')}")
print(f"XXXX  (code inexistant) → {transcoder_avec_troncature('XXXX')}")

=== Test normalisation ===
C180  → C18.0
S7200 → S72.00
M23   → M23
C18.0 → C18.0

=== Test transcodage ===
C180  (code exact)      → 2B90.0Z&XA6J68
S7200 (troncature 1)    → NC72.2Z
S72   (sans décimale)   → NC72.Z
XXXX  (code inexistant) → None


## 11. Application du transcodage sur les scénarios
Application du transcodage CIM-10 → CIM-11 sur les diagnostics principaux et secondaires.
Les lignes sans équivalent CIM-11 sont supprimées (environ 0.6% des scénarios).

> **MODIF CIM-11** : Cellule ajoutée — enrichit `gs.df_classification_profile` avec les colonnes `icd11_primary_code` et `icd11_secondary_code`.

In [14]:
# MODIF CIM-11 : Transcodage du diagnostic principal CIM-10 → CIM-11
gs.df_classification_profile['icd11_primary_code'] = gs.df_classification_profile['icd_primary_code'].apply(
    transcoder_avec_troncature
)

# MODIF CIM-11 : Suppression des lignes sans équivalent CIM-11
# Ces lignes (0.6%) correspondent à des codes trop spécifiques absents de la table OMS
nb_avant = len(gs.df_classification_profile)
gs.df_classification_profile = gs.df_classification_profile[
    gs.df_classification_profile['icd11_primary_code'].notna()
].reset_index(drop=True)
nb_apres = len(gs.df_classification_profile)
print(f'Scénarios conservés : {nb_apres} / {nb_avant} ({nb_apres/nb_avant*100:.1f}%)')

# MODIF CIM-11 : Suppression des codes résiduels (dernier caractère Y ou Z)
# Y = "autre affection précisée", Z = "affection sans précision" — pas assez
# spécifiques pour générer un scénario clinique cohérent
nb_avant_yz = len(gs.df_classification_profile)
gs.df_classification_profile = gs.df_classification_profile[
    ~gs.df_classification_profile['icd11_primary_code'].str.endswith(('Y', 'Z'))
].reset_index(drop=True)
nb_apres_yz = len(gs.df_classification_profile)
print(f'Scénarios conservés après filtrage Y/Z : {nb_apres_yz} / {nb_avant_yz} ({nb_apres_yz/nb_avant_yz*100:.1f}%)')

# MODIF CIM-11 : Transcodage du diagnostic secondaire CIM-10 → CIM-11
# La colonne icd_secondary_code contient des listes (plusieurs diagnostics secondaires possibles)
# d'où la fonction dédiée qui traite les listes et les valeurs simples
def transcoder_secondaire(x):
    if x is None:
        return None
    if isinstance(x, list):  # Cas liste : on transcorde chaque code
        codes = [transcoder_avec_troncature(str(code).strip()) for code in x if str(code).strip() != '']
        # On retire les codes résiduels (Y/Z) et les échecs de transcodage (None)
        return [c for c in codes if c is not None and not c.endswith(('Y', 'Z'))]
    if pd.isna(x) or str(x).strip() == '':  # Cas vide
        return None
    code = transcoder_avec_troncature(str(x).strip())
    if code is not None and code.endswith(('Y', 'Z')):
        return None
    return code

gs.df_classification_profile['icd11_secondary_code'] = gs.df_classification_profile['icd_secondary_code'].apply(transcoder_secondaire)
print('Transcodage secondaire terminé !')

# Vérification du transcodage sur les 5 premières lignes
print(f"\nExemple de transcodage :")
print(gs.df_classification_profile[['icd_primary_code', 'icd11_primary_code']].head(5))

Scénarios conservés : 148931 / 149014 (99.9%)
Scénarios conservés après filtrage Y/Z : 75927 / 148931 (51.0%)
Transcodage secondaire terminé !

Exemple de transcodage :
  icd_primary_code icd11_primary_code
0             C220            2C12.02
1             N185             GB61.5
2             O800             JB20.0
3             C795               2E03
4             O800             JB20.0


### Test — Taux de conversion du transcodage
Vérifie le taux de conversion et montre les codes non convertis.

In [15]:
# Taux de conversion global
total = len(gs.df_classification_profile)
convertis_p = gs.df_classification_profile['icd11_primary_code'].notna().sum()
convertis_s = gs.df_classification_profile['icd11_secondary_code'].notna().sum()
print(f"Diagnostics principaux convertis : {convertis_p}/{total} ({convertis_p/total*100:.1f}%)")
print(f"Diagnostics secondaires convertis : {convertis_s} (sur ceux qui en ont)")

# Exemple de transcodage sur les 5 premières lignes
print("\nExemple principal :")
print(gs.df_classification_profile[['icd_primary_code', 'icd11_primary_code']].head(5))

# Exemple de transcodage secondaire (chercher une ligne avec secondaire)
mask = gs.df_classification_profile['icd11_secondary_code'].apply(
    lambda x: isinstance(x, list) and len(x) > 0 and x[0] is not None
)
if mask.any():
    exemple = gs.df_classification_profile[mask][['icd_secondary_code', 'icd11_secondary_code']].head(3)
    print("\nExemple secondaire :")
    print(exemple)

Diagnostics principaux convertis : 75927/75927 (100.0%)
Diagnostics secondaires convertis : 75927 (sur ceux qui en ont)

Exemple principal :
  icd_primary_code icd11_primary_code
0             C220            2C12.02
1             N185             GB61.5
2             O800             JB20.0
3             C795               2E03
4             O800             JB20.0

Exemple secondaire :
                                  icd_secondary_code  \
0  [K703, K746, B181, E1198, F10202, G473, K758, ...   
1              [Z290, Z713, I770, Z292, Z4580, Z488]   
2  [O431, Z370, D259, J450, O234, O702, O720, Z59...   

                                icd11_secondary_code  
0                             [DB94.3, DB93.1, 5A11]  
1                             [QC05.0, QA10, BD52.1]  
2  [JA8A.1, QA46.0, 2E86.0, CA23.02, JB09.2, JA43...  


## 12. Chargement des libellés CIM-11 et des templates
> **MODIF CIM-11** : Cellule ajoutée — quatre opérations :
> 1. **Libellés CIM-11** : chargés depuis `cim11_termes.csv`, filtrés sur `type == 'title'` pour avoir uniquement les libellés officiels (pas les synonymes)
> 2. **Templates** : remplacement de `CIM-10` et `CIM10` par `CIM-11` dans tous les fichiers `.txt`
> 3. **Instruction postcoordination** : Mistral doit choisir UN SEUL code post-coordonné parmi la liste fournie, tel quel
> 4. **Instruction éléments postcoordination** : Mistral doit décrire cliniquement chaque élément du code post-coordonné dans le CRH (ex: classe NYHA, caractère chronique...)

In [16]:
# MODIF CIM-11 : Chargement des libellés officiels CIM-11
# On filtre sur type == 'title' pour avoir uniquement le libellé principal
# (et non les synonymes, inclusions, etc.)
df_syn = pd.read_csv(
    '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/data/cim11_termes.csv',
    encoding='utf-8-sig'
)
df_syn_title = df_syn[df_syn['type'] == 'title']
dict_cim11_libelles = dict(zip(df_syn_title['code'], df_syn_title['texte']))
print(f'Dictionnaire CIM-11 : {len(dict_cim11_libelles)} entrées')

# MODIF CIM-11 : Chargement des templates avec adaptation CIM-11
# Les templates originaux mentionnent CIM-10 → on remplace par CIM-11
# On ajoute aussi des instructions pour que Mistral utilise et décrive les codes post-coordonnés
template_dir = '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/templates'
templates = {}
for fichier in os.listdir(template_dir):
    if fichier.endswith('.txt'):
        nom_variable = fichier.replace('.txt', '')
        with open(os.path.join(template_dir, fichier), 'r', encoding='utf-8') as f:
            contenu = f.read()

        # Remplacement de toutes les occurrences de CIM-10 par CIM-11
        contenu = contenu.replace('CIM-10', 'CIM-11')
        contenu = contenu.replace('CIM10', 'CIM-11')  # Version sans tiret

        # Instruction pour que Mistral utilise le code post-coordonné fourni tel quel
        # Sans cette instruction, Mistral tend à créer de nouveaux codes ou en utiliser plusieurs
        instruction_postcoord = "\n- Lorsque des codes post-coordonnés sont fournis dans le scénario, vous devez OBLIGATOIREMENT choisir UN SEUL code post-coordonné parmi la liste fournie, tel quel, sans le modifier ni en créer un nouveau. Utilisez ce code exact dans le dictionnaire de formulations des diagnostics.\n"

        # MODIF CIM-11 : Instruction pour décrire cliniquement chaque élément de la postcoordination
        # Chaque extension du code post-coordonné doit être mentionnée explicitement dans le CRH
        # Ex: pour 'BD10 & XS9F & XT8W' (Insuffisance cardiaque - NYHA classe IV - Chronique),
        # le CRH doit mentionner la classe NYHA IV et le caractère chronique
        instruction_elements_postcoord = "\n- Lorsqu'un code post-coordonné est fourni, vous devez décrire cliniquement dans le texte du CRH chaque élément de ce code. Par exemple pour 'Insuffisance cardiaque - NYHA classe IV - Chronique (BD10 & XS9F & XT8W)', le CRH doit mentionner explicitement la classe NYHA IV et le caractère chronique.\n"

        contenu = contenu.replace(
            "- Vous devez utiliser exclusivement les diagnostics fournis.",
            "- Vous devez utiliser exclusivement les diagnostics fournis." + instruction_postcoord + instruction_elements_postcoord
        )
        templates[nom_variable] = contenu

print(f'Templates chargés : {list(templates.keys())}')

# Vérification qu'il n'y a plus de CIM-10 dans les templates
for nom, contenu in templates.items():
    if 'CIM-10' in contenu or 'CIM10' in contenu:
        print(f"CIM-10 encore présent dans : {nom}")
    else:
        print(f"{nom} — OK")

Dictionnaire CIM-11 : 34663 entrées
Templates chargés : ['delivery_inpatient_csection_urg', 'medical_outpatient_onco', 'medical_inpatient_onco', 'surgery_outpatient', 'surgery_outpatient_onco', 'delivery_inpatient_hospit', 'surgery_inpatient_onco', 'delivery_inpatient_csection_hospit', 'medical_outpatient', 'medical_inpatient_obsteric', 'medical_inpatient', 'delivery_inpatient_urg', 'surgery_inpatient']
delivery_inpatient_csection_urg — OK
medical_outpatient_onco — OK
medical_inpatient_onco — OK
surgery_outpatient — OK
surgery_outpatient_onco — OK
delivery_inpatient_hospit — OK
surgery_inpatient_onco — OK
delivery_inpatient_csection_hospit — OK
medical_outpatient — OK
medical_inpatient_obsteric — OK
medical_inpatient — OK
delivery_inpatient_urg — OK
surgery_inpatient — OK


### Test — Dictionnaire CIM-11 et templates
Vérifie que les libellés sont bien chargés et que les templates ne contiennent plus de CIM-10.

In [17]:
# Test du dictionnaire CIM-11
print("=== Test dictionnaire CIM-11 ===")
codes_test = ['BD10', 'DA22.Z', 'CA81.Z', '5A11']
for code in codes_test:
    libelle = dict_cim11_libelles.get(code, 'NON TROUVÉ')
    print(f"{code} → {libelle}")

# Test des templates : vérifier que CIM-10 a bien été remplacé
print("\n=== Vérification des templates ===")
for nom, contenu in templates.items():
    nb_cim10 = contenu.count('CIM-10') + contenu.count('CIM10')
    nb_cim11 = contenu.count('CIM-11')
    status = 'OK' if nb_cim10 == 0 else 'ATTENTION'
    print(f"{status} - {nom} : {nb_cim11} occurrences CIM-11, {nb_cim10} occurrences CIM-10")

=== Test dictionnaire CIM-11 ===
BD10 → Insuffisance cardiaque congestive
DA22.Z → Reflux gastro-œsophagien, sans précision
CA81.Z → Maladies respiratoires dues à l'inhalation de produits chimiques, de gaz, de fumées ou de vapeurs, sans précision
5A11 → Diabète sucré de type 2

=== Vérification des templates ===
OK - delivery_inpatient_csection_urg : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - medical_outpatient_onco : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - medical_inpatient_onco : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - surgery_outpatient : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - surgery_outpatient_onco : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - delivery_inpatient_hospit : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - surgery_inpatient_onco : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - delivery_inpatient_csection_hospit : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - medical_outpatient : 4 occurrences CIM-11, 0 occurrences CIM-10
OK - medica

## 13. Chargement des codes post-coordonnés et fonctions associées
> **MODIF CIM-11** : Cellule ajoutée.
>
> **Contexte** : En CIM-11, un code peut être enrichi avec des extensions (postcoordination) via la syntaxe `code & extension`. Ex: `BD10 & XS9F` = Insuffisance cardiaque + NYHA classe IV.
>
> Le fichier `cim11_postcoord.csv` contient toutes les combinaisons valides générées par `colab_generation.ipynb`.
>
> **Fonction `tirer_codes_postcoord()`** : pour un code CIM-11, cherche tous les codes post-coordonnés disponibles et en tire au sort n.
>
> **Fonction `verifier_coherence()`** : demande à Mistral si un code post-coordonné est cohérent avec le patient (âge, sexe). `max_tokens=5` limite la réponse à "OUI" ou "NON" pour minimiser le coût.

In [18]:
# MODIF CIM-11 : Chargement du fichier de codes post-coordonnés réalistes v2 (filtrés par Mistral)
df_postcoord = pd.read_csv(
    '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/data/codes_postcoord_realistes_v2.csv'
)
print(f"Codes post-coordonnés chargés : {df_postcoord.shape}")
print(df_postcoord.head(3))

# MODIF CIM-11 : Dictionnaire code_postcoord -> libellé complet (racine + extensions)
# Évite de refiltrer le DataFrame à chaque appel (accès O(1))
dict_libelle_postcoord = {}
for _, r in df_postcoord.drop_duplicates('code_postcoord').iterrows():
    if pd.notna(r['libelles_extensions']) and str(r['libelles_extensions']).strip():
        dict_libelle_postcoord[r['code_postcoord']] = f"{r['libelle_racine']} - {r['libelles_extensions']}"
    else:
        dict_libelle_postcoord[r['code_postcoord']] = r['libelle_racine']

print(f"Libellés post-coordonnés indexés : {len(dict_libelle_postcoord)}")


def tirer_codes_postcoord(code_cim11, df_postcoord, n=5):
    """
    Pour un code CIM-11 donné, tire au sort n codes post-coordonnés
    parmi ceux disponibles dans df_postcoord.
    Retourne une liste vide si aucun code post-coordonné n'est disponible.
    """
    candidats = df_postcoord[
        df_postcoord['code_racine'] == str(code_cim11)
    ]['code_postcoord'].tolist()

    if not candidats:
        return []

    n_tirage = min(n, len(candidats))
    return random.sample(candidats, n_tirage)


def get_libelle_postcoord(code_postcoord, libelle_racine_fallback=''):
    """
    Retourne le libellé complet d'un code post-coordonné (racine + extensions).
    Si le code n'est pas trouvé, retourne le libellé de la racine fourni en fallback.
    """
    return dict_libelle_postcoord.get(code_postcoord, libelle_racine_fallback)

Codes post-coordonnés chargés : (593652, 4)
  code_racine                                     libelle_racine  \
0      1A62.2  Syphilis tardive symptomatique d'autres locali...   
1      1A62.2  Syphilis tardive symptomatique d'autres locali...   
2      1A62.2  Syphilis tardive symptomatique d'autres locali...   

  code_postcoord                   libelles_extensions  
0  1A62.2&XA6GV0                      XA6GV0 : Abdomen  
1  1A62.2&XA3KX0             XA3KX0 : Paroi abdominale  
2  1A62.2&XA4SN6  XA4SN6 : Paroi abdominale antérieure  
Libellés post-coordonnés indexés : 531221


In [19]:
import time

# MODIF CIM-11 : Vérification de cohérence patient/code via Mistral
# Pour chaque code post-coordonné tiré au sort, on demande à Mistral si ce code
# est MÉDICALEMENT IMPOSSIBLE pour ce patient (âge, sexe)
# On utilise une formulation négative ("impossible ?") plutôt que positive ("cohérent ?")
# car Mistral était trop strict avec la formulation positive et refusait des codes valides
# (ex: localisation anatomique refusée pour un patient adulte)
# max_tokens=5 limite la réponse à "OUI" ou "NON" pour minimiser le coût API
# time.sleep(0.5) évite le rate limit (erreur 429) de l'API Mistral
# En cas d'erreur API, on considère le code comme cohérent pour ne pas bloquer le pipeline
def verifier_coherence(code, libelle, age, sexe, client, model):
    """
    Vérifie via Mistral si un code post-coordonné est médicalement impossible pour ce patient.
    Retourne True si le code est acceptable, False s'il est impossible.
    """
    sexe_texte = "Masculin" if sexe == 1 else "Féminin"
    prompt = f"""Patient : {age} ans, {sexe_texte}
Code post-coordonné CIM-11 : {code}
Libellé : {libelle}

Ce code post-coordonné est-il MÉDICALEMENT IMPOSSIBLE pour ce patient ?
Réponds NON uniquement si c'est impossible (ex: grossesse pour un homme, période néonatale pour un adulte, histologie pédiatrique pour un senior).
Réponds OUI dans tous les autres cas, y compris les localisations anatomiques.
Réponds UNIQUEMENT par OUI ou NON, sans aucune explication."""

    try:
        time.sleep(0.5)  # Délai pour éviter le rate limit de l'API Mistral (erreur 429)
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5
        )
        reponse = response.choices[0].message.content.strip().upper()
        return reponse == "OUI"
    except Exception as e:
        print(f"Erreur vérification cohérence : {e}")
        return True


# MODIF CIM-11 : Vérification de cohérence entre un DAS et le diagnostic principal
# On utilise une formulation négative ("impossible ?") pour éviter que Mistral soit trop strict
def verifier_coherence_das_dp(code_das, libelle_das, code_dp, libelle_dp, client, model):
    """
    Vérifie via Mistral si un DAS est cohérent avec le diagnostic principal.
    Retourne True si cohérent, False sinon.
    """
    prompt = f"""Diagnostic principal : {libelle_dp} (CIM-11 : {code_dp})
Diagnostic associé : {libelle_das} (CIM-11 : {code_das})

Ce diagnostic associé est-il MÉDICALEMENT IMPOSSIBLE en association avec le diagnostic principal ?
Réponds NON uniquement si c'est impossible (ex: deux maladies qui s'excluent mutuellement).
Réponds OUI dans tous les autres cas.
Réponds UNIQUEMENT par OUI ou NON, sans aucune explication."""

    try:
        time.sleep(0.5)  # Délai pour éviter le rate limit
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5
        )
        reponse = response.choices[0].message.content.strip().upper()
        return reponse == "OUI"
    except Exception as e:
        print(f"Erreur vérification cohérence DAS/DP : {e}")
        return True


# MODIF CIM-11 : Vérification de cohérence entre tous les DAS entre eux et avec le DP
# max_tokens=50 car Mistral doit lister les codes à supprimer
def verifier_coherence_das_entre_eux(das_list, code_dp, libelle_dp, client, model):
    """
    Vérifie via Mistral si l'ensemble des DAS est cohérent entre eux et avec le DP.
    Retourne la liste des DAS cohérents (code, libelle).
    """
    if len(das_list) <= 1:
        return das_list

    das_texte = "\n".join([f"- {libelle} (CIM-11 : {code})" for code, libelle in das_list])

    prompt = f"""Diagnostic principal : {libelle_dp} (CIM-11 : {code_dp})
Diagnostics associés :
{das_texte}

Parmi ces diagnostics associés, lesquels sont MÉDICALEMENT IMPOSSIBLES en association avec le diagnostic principal ou entre eux ?
Réponds avec uniquement les codes CIM-11 des diagnostics impossibles, séparés par des virgules.
Si tous sont cohérents, réponds AUCUN.
Ne donne aucune explication."""

    try:
        time.sleep(0.5)  # Délai pour éviter le rate limit
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        reponse = response.choices[0].message.content.strip().upper()
        if reponse == "AUCUN":
            return das_list
        codes_a_supprimer = [c.strip() for c in reponse.split(',')]
        return [(code, libelle) for code, libelle in das_list if code not in codes_a_supprimer]
    except Exception as e:
        print(f"Erreur vérification cohérence DAS entre eux : {e}")
        return das_list

In [20]:
# MODIF CIM-11 : Vérification de cohérence entre un DAS et le diagnostic principal
# On utilise une formulation négative ("impossible ?") pour éviter que Mistral soit trop strict
def verifier_coherence_das_dp(code_das, libelle_das, code_dp, libelle_dp, client, model):
    """
    Vérifie via Mistral si un DAS est cohérent avec le diagnostic principal.
    Retourne True si cohérent, False sinon.
    """
    prompt = f"""Diagnostic principal : {libelle_dp} (CIM-11 : {code_dp})
Diagnostic associé : {libelle_das} (CIM-11 : {code_das})

Ce diagnostic associé est-il MÉDICALEMENT IMPOSSIBLE en association avec le diagnostic principal ?
Réponds NON uniquement si c'est impossible (ex: deux maladies qui s'excluent mutuellement).
Réponds OUI dans tous les autres cas.
Réponds UNIQUEMENT par OUI ou NON, sans aucune explication."""

    try:
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5
        )
        reponse = response.choices[0].message.content.strip().upper()
        return reponse == "OUI"
    except Exception as e:
        print(f"Erreur vérification cohérence DAS/DP : {e}")
        return True


# MODIF CIM-11 : Vérification de cohérence entre tous les DAS entre eux et avec le DP
# max_tokens=50 car Mistral doit lister les codes à supprimer
def verifier_coherence_das_entre_eux(das_list, code_dp, libelle_dp, client, model):
    """
    Vérifie via Mistral si l'ensemble des DAS est cohérent entre eux et avec le DP.
    Retourne la liste des DAS cohérents (code, libelle).
    """
    if len(das_list) <= 1:
        return das_list

    das_texte = "\n".join([f"- {libelle} (CIM-11 : {code})" for code, libelle in das_list])

    prompt = f"""Diagnostic principal : {libelle_dp} (CIM-11 : {code_dp})
Diagnostics associés :
{das_texte}

Parmi ces diagnostics associés, lesquels sont MÉDICALEMENT IMPOSSIBLES en association avec le diagnostic principal ou entre eux ?
Réponds avec uniquement les codes CIM-11 des diagnostics impossibles, séparés par des virgules.
Si tous sont cohérents, réponds AUCUN.
Ne donne aucune explication."""

    try:
        response = client.chat.complete(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        reponse = response.choices[0].message.content.strip().upper()
        if reponse == "AUCUN":
            return das_list
        codes_a_supprimer = [c.strip() for c in reponse.split(',')]
        return [(code, libelle) for code, libelle in das_list if code not in codes_a_supprimer]
    except Exception as e:
        print(f"Erreur vérification cohérence DAS entre eux : {e}")
        return das_list

## 14. Sampling des scénarios
Tirage au sort de `N_SCENARIO` lignes parmi les scénarios disponibles, avec pondération par fréquence (`weights="nb"`).

La pondération garantit que les séjours les plus fréquents dans les données PMSI sont représentés proportionnellement.

In [21]:
# Filtrage optionnel selon QUERY (ex: exclure les cancers)
if QUERY is not None:
    df_profile_sample = gs.df_classification_profile.query(QUERY)
else:
    df_profile_sample = gs.df_classification_profile.copy()

# Tirage au sort pondéré par la fréquence (colonne 'nb')
df_profile_sample = df_profile_sample.sample(N_SCENARIO, weights="nb").reset_index(drop=True)

# Vérification de l'échantillon
print(f"Scénarios tirés au sort : {len(df_profile_sample)}")
print(f"Répartition des types d'hospitalisation :")
print(df_profile_sample['admission_type'].value_counts())

Scénarios tirés au sort : 200
Répartition des types d'hospitalisation :
admission_type
Inpatient     166
Outpatient     34
Name: count, dtype: int64


###  Test — Vérification de l'échantillon
Vérifie la distribution de l'échantillon tiré au sort.

In [22]:
# Distribution des codes CIM-11 dans l'échantillon
print("=== Répartition des types d'hospitalisation ===")
print(df_profile_sample['admission_type'].value_counts())

print("\n=== Exemples de codes CIM-10 et CIM-11 ===")
print(df_profile_sample[['icd_primary_code', 'icd11_primary_code', 'age', 'sexe']].head(5))

# Pourcentage avec diagnostic secondaire
avec_sec = df_profile_sample['icd11_secondary_code'].apply(
    lambda x: isinstance(x, list) and len(x) > 0 and any(v is not None for v in x)
).sum()
print(f"\nScénarios avec diagnostic secondaire : {avec_sec}/{N_SCENARIO} ({avec_sec/N_SCENARIO*100:.0f}%)")

=== Répartition des types d'hospitalisation ===
admission_type
Inpatient     166
Outpatient     34
Name: count, dtype: int64

=== Exemples de codes CIM-10 et CIM-11 ===
  icd_primary_code icd11_primary_code    age sexe
0             O800             JB20.0  ge_18    2
1             O610             JB01.0  ge_18    2
2             O800             JB20.0  ge_18    2
3             O600             JB00.0  ge_18    2
4             A870             1C8E.1  lt_18    1

Scénarios avec diagnostic secondaire : 157/200 (78%)


## 15b. Configuration de l'API Mistral
La configuration doit être faite **avant** la cellule de génération car `verifier_coherence()` utilise `client` et `model`.

In [23]:
from config import *
api_url = "https://api.mistral.ai/v1/chat"
model = "mistral-large-latest"
client = Mistral(api_key=api_key)

In [24]:
# Nom du fichier de sortie final
scenario_file_name = "generated_scenarios_cim11"

# Taille des batches envoyés à Mistral
# L'API Mistral accepte au maximum 100 requêtes par batch
n_samples_by_batch = 100

## 16. Génération des scénarios CIM-11 avec codes post-coordonnés
Pour chaque scénario tiré au sort :
1. Génération du scénario clinique via `generate_scenario_from_profile()`
2. Ajout des codes et libellés CIM-11
3. Pour chaque DAS : tirage d'1 code post-coordonné + vérification cohérence avec le DP → filtrés dans `icd11_secondary_code`
4. Vérification cohérence entre tous les DAS restants
5. Pour le DP : tirage de 5 codes post-coordonnés cohérents → 1 prompt par code
6. Construction du prompt final avec le template CIM-11

> **MODIF CIM-11** : Cellule fortement modifiée par rapport à l'original.
>
> **DAS** : chaque DAS reçoit un code post-coordonné tiré au sort. Si non cohérent avec le DP → retiré. À la fin, cohérence vérifiée entre tous les DAS restants. Deux colonnes : `icd11_secondary_code_all` (tous) et `icd11_secondary_code` (filtrés).
>
> **DP** : tirage de 5 codes post-coordonnés, vérification cohérence patient (âge, sexe). Un prompt par code cohérent → une ligne par CRH dans le fichier final.

In [25]:
list_scenario = []

for i in tqdm(range(len(df_profile_sample))):
    profile = df_profile_sample.iloc[i].copy()

    # Génération du scénario clinique (noms fictifs, dates, contexte hospitalier...)
    scenario = gs.generate_scenario_from_profile(profile)
    row = {k: scenario[k] for k in scenario.keys()}

    # MODIF CIM-11 : Ajout des codes et libellés CIM-11 dans le dictionnaire scenario
    # Nécessaire car make_prompts_marks_from_scenario() lit depuis ce dictionnaire
    code_cim10 = scenario['icd_primary_code']
    code_cim11 = profile.get('icd11_primary_code', None)
    libelle_cim11 = dict_cim11_libelles.get(str(code_cim11), '') if code_cim11 else ''

    scenario['icd11_primary_code'] = code_cim11
    scenario['icd11_primary_description'] = libelle_cim11
    row['icd11_primary_code'] = code_cim11
    row['icd11_primary_description'] = libelle_cim11

    # MODIF CIM-11 : Tirage au sort d'un code post-coordonné pour chaque DAS
    # et vérification de cohérence avec le DP
    # das_all : tous les DAS transcodés (avant filtrage) → sauvegardé dans icd11_secondary_code_all
    # das_gardes : DAS cohérents avec le DP (après filtrage) → sauvegardé dans icd11_secondary_code
    sec_code = scenario.get('icd_secondary_code')
    das_all = []
    das_gardes = []

    if sec_code and isinstance(sec_code, list) and len(sec_code) > 0:
        code_sec_cim11_list = profile.get('icd11_secondary_code', None)
        if code_sec_cim11_list and isinstance(code_sec_cim11_list, list):
            for code_sec in code_sec_cim11_list:
                if not code_sec:
                    continue
                libelle_sec = dict_cim11_libelles.get(str(code_sec), '')
                das_all.append((code_sec, libelle_sec))

                # Tirage au sort d'UN seul code post-coordonné pour ce DAS
                candidats_sec = df_postcoord[
                    df_postcoord['code_racine'] == str(code_sec)
                ]['code_postcoord'].tolist()
                candidats_sec = list(dict.fromkeys(candidats_sec))

                if candidats_sec:
                    code_postcoord_sec = random.choice(candidats_sec)
                    libelle_postcoord_sec = get_libelle_postcoord(code_postcoord_sec, libelle_sec)
                else:
                    code_postcoord_sec = code_sec
                    libelle_postcoord_sec = libelle_sec

                # Vérification cohérence DAS avec le DP
                if verifier_coherence_das_dp(code_postcoord_sec, libelle_postcoord_sec, code_cim11, libelle_cim11, client, model):
                    das_gardes.append((code_postcoord_sec, libelle_postcoord_sec))

    # Vérification cohérence entre tous les DAS restants entre eux et avec le DP
    if len(das_gardes) > 1:
        das_gardes = verifier_coherence_das_entre_eux(das_gardes, code_cim11, libelle_cim11, client, model)

    # Construction du texte des DAS pour le prompt
    if das_gardes:
        scenario['text_secondary_icd_official'] = '\n'.join([f"- {libelle} (CIM-11 : {code})" for code, libelle in das_gardes])
    else:
        scenario['text_secondary_icd_official'] = ''

    # Sauvegarde des DAS complets et filtrés pour le fichier de sortie
    row['icd11_secondary_code_all'] = [code for code, _ in das_all]
    row['icd11_secondary_code'] = [code for code, _ in das_gardes]

    # MODIF CIM-11 : Génération des prompts avec vérification de cohérence via Mistral
    # On tire au sort 5 codes post-coordonnés parmi tous les disponibles
    # Pour chaque code, Mistral vérifie si c'est cohérent avec le patient (âge, sexe)
    # On garde les codes cohérents (jusqu'à 5) et on génère un prompt par code
    user_prompts = []
    codes_postcoord_all = df_postcoord[
        df_postcoord['code_racine'] == str(code_cim11)
    ]['code_postcoord'].tolist()
    codes_postcoord_all = list(dict.fromkeys(codes_postcoord_all))

    if not codes_postcoord_all:
        codes_a_utiliser = [(code_cim11, libelle_cim11)]
    else:
        codes_a_utiliser = []
        selection = random.sample(codes_postcoord_all, min(5, len(codes_postcoord_all)))

        for code in selection:
            libelle_texte = get_libelle_postcoord(code, libelle_cim11)

            if verifier_coherence(code, libelle_texte, profile['age'], profile['sexe'], client, model):
                codes_a_utiliser.append((code, libelle_texte))

            if len(codes_a_utiliser) == 5:
                break

        if not codes_a_utiliser:
            codes_a_utiliser = [(code_cim11, libelle_cim11)]

    # Génération d'un prompt par code post-coordonné cohérent
    for j, (code, libelle_texte) in enumerate(codes_a_utiliser):
        scenario_copy = scenario.copy()
        scenario_copy['codes_postcoord_libelles'] = [(code, libelle_texte)]
        scenario_copy['codes_postcoord'] = [code]
        user_prompts.append(gs.make_prompts_marks_from_scenario(scenario_copy))

    # MODIF CIM-11 : Sauvegarde du code et libellé post-coordonné réellement utilisés,
    # un par prompt, pour traçabilité dans le fichier final
    row['icd11_primary_code_postcoord_list'] = [c for c, _ in codes_a_utiliser]
    row['icd11_primary_libelle_postcoord_list'] = [l for _, l in codes_a_utiliser]

    # Stockage des prompts avec numérotation dynamique
    for j, prompt in enumerate(user_prompts):
        row[f'user_prompt_{j+1}'] = prompt

    # MODIF CIM-11 : Utilisation des templates chargés en mémoire (avec CIM-10 → CIM-11)
    template_name = scenario['template_name'].replace('.txt', '')
    system_prompt = templates.get(template_name, templates['medical_inpatient'])

    if scenario["icd_primary_code"] in gs.icd_codes_cancer:
        prefix = """Le compte rendu suivant respecte les élements suivants :
        - les diagnostics ont une formulation moins formelle que la définition du code
        - le type histologique et la valeur des biomarqueurs si recherchés
        - le plan du CRH est conforme aux recommandations.
        """
    else:
        prefix = """Le compte rendu suivant respecte les élements suivants :
        - les diagnostics ont une formulation moins formelle que la définition du code
        - le plan du CRH est conforme aux recommandations.
        """

    row["system_prompt"] = system_prompt
    row["prefix"] = prefix
    row["prefix_len"] = len(prefix)
    list_scenario.append(row)

# Vérification des 3 premiers scénarios
for i, scenario in enumerate(list_scenario[:3]):
    print(f"\n{'='*50}")
    print(f"SCÉNARIO {i+1}")
    print(f"CIM-10 : {scenario['icd_primary_code']} → CIM-11 : {scenario['icd11_primary_code']}")
    print(f"Nb prompts générés : {sum(1 for k in scenario if k.startswith('user_prompt_'))}")
    print(f"DAS all : {scenario.get('icd11_secondary_code_all', [])}")
    print(f"DAS gardés : {scenario.get('icd11_secondary_code', [])}")
    print(f"Template : {scenario['template_name']}")

  2%|▏         | 3/200 [00:10<10:58,  3.34s/it]

Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


  2%|▏         | 4/200 [00:14<12:54,  3.95s/it]

Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


  2%|▎         | 5/200 [00:19<13:14,  4.08s/it]

...

Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : A

 99%|█████████▉| 198/200 [10:45<00:04,  2.41s/it]

Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS entre eux : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence DAS/DP : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API error occurred: Status 429
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Erreur vérification cohérence : API erro

100%|██████████| 200/200 [10:57<00:00,  3.29s/it]


SCÉNARIO 1
CIM-10 : O800 → CIM-11 : JB20.0
Nb prompts générés : 1
DAS all : ['QA46.0', 'QA48.1', 'JA61.0']
DAS gardés : ['QA46.0', 'QA48.1', 'JA61.0&XK9J&XK70']
Template : delivery_inpatient_hospit.txt

SCÉNARIO 2
CIM-10 : O610 → CIM-11 : JB01.0
Nb prompts générés : 1
DAS all : ['JA8E', 'QA46.0', 'QA48.1']
DAS gardés : ['JA8E', 'QA46.0', 'QA48.1']
Template : delivery_inpatient_csection_hospit.txt

SCÉNARIO 3
CIM-10 : O800 → CIM-11 : JB20.0
Nb prompts générés : 1
DAS all : ['JA63.2', 'QA21.1', 'QA46.0', 'QA48.1']
DAS gardés : ['JA63.2', 'QA21.1', 'QA46.0', 'QA48.1']
Template : delivery_inpatient_hospit.txt


###  Test — Cohérence de la durée de séjour
Vérifie que la durée calculée depuis les dates d'entrée/sortie correspond à la durée dans le fichier Parquet.

In [26]:
# Test : vérification de la cohérence de la durée de séjour
print("=== Vérification durée de séjour ===")
for i, scenario in enumerate(list_scenario[:5]):
    date_entree = pd.to_datetime(scenario['date_entry'], dayfirst=True)
    date_sortie = pd.to_datetime(scenario['date_discharge'], dayfirst=True)
    duree_calculee = (date_sortie - date_entree).days
    duree_parquet = scenario.get('los', 'N/A')
    coherent = "OK" if duree_calculee == duree_parquet else "ATTENTION"
    print(f"Scénario {i+1} : entrée={scenario['date_entry']} sortie={scenario['date_discharge']} "
          f"| durée calculée={duree_calculee}j | durée Parquet={duree_parquet}j | {coherent}")

=== Vérification durée de séjour ===
Scénario 1 : entrée=2024-02-13 sortie=2024-02-17 | durée calculée=4j | durée Parquet=4.0j | OK
Scénario 2 : entrée=2026-01-29 sortie=2026-02-05 | durée calculée=7j | durée Parquet=7.0j | OK
Scénario 3 : entrée=2024-05-10 sortie=2024-05-15 | durée calculée=5j | durée Parquet=5.0j | OK
Scénario 4 : entrée=2024-12-05 sortie=2024-12-16 | durée calculée=11j | durée Parquet=11.0j | OK
Scénario 5 : entrée=2025-03-17 sortie=2025-03-21 | durée calculée=4j | durée Parquet=4.0j | OK


###  Test — Tirage au sort de codes post-coordonnés
Vérifie que la fonction de tirage retourne bien des codes pour un code CIM-11 donné.

In [27]:
# Test de tirer_codes_postcoord
print("=== Test tirage au sort ===")

code_test = 'BD10'
candidats_total = df_postcoord[df_postcoord['code_racine'] == code_test]
print(f"'{code_test}' : {len(candidats_total)} codes disponibles")
codes = tirer_codes_postcoord(code_test, df_postcoord, n=3)
print(f"Tirage de 3 codes :")
for code in codes:
    print(f"  → {code}")

code_test2 = 'XXXXX'
codes2 = tirer_codes_postcoord(code_test2, df_postcoord, n=5)
print(f"\n'{code_test2}' (inexistant) : {codes2}")

=== Test tirage au sort ===
'BD10' : 140 codes disponibles
Tirage de 3 codes :
  → BD10&XS9T&XT8W
  → BD10&XS9F
  → BD10&XS9T&XT8W

'XXXXX' (inexistant) : []


## 17. Sauvegarde des scénarios générés
Création d'un dossier horodaté et sauvegarde des scénarios avec leurs prompts.

> **MODIF CIM-11** : Cellule modifiée.
>
> **Nouveau fichier** : suppression de `cage`, `los_mean`, `los_sd` absentes du nouveau fichier ; ajout de `age2` (âge numérique) et `los` (durée de séjour).
>
> **Une ligne par CRH** : on garde uniquement `user_prompt_1` dans `keep_cols` — chaque prompt correspond à un code post-coordonné différent et générera un CRH distinct.

In [28]:
# MODIF CIM-11 : Création d'un dossier horodaté pour cette exécution
# Format : YYYYMMDD_HHMMSS pour garantir l'unicité
run_id = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = path_results + run_id + "/"
os.makedirs(run_dir + "tmp/", exist_ok=True)
print(f"Dossier de sortie : {run_dir}")

# MODIF CIM-11 : Colonnes à conserver dans le fichier de sortie
# MODIF nouveau fichier : suppression de cage, los_mean, los_sd (absentes du nouveau fichier)
# MODIF nouveau fichier : ajout de age2 (âge numérique) et los (durée de séjour)
# On garde uniquement les colonnes user_prompt qui existent réellement
# (certains runs n'ont pas 5 prompts par séjour)
df_list = pd.DataFrame(list_scenario)
user_prompt_cols = [f'user_prompt_{i}' for i in range(1, 6) if f'user_prompt_{i}' in df_list.columns]

keep_cols = ['age', 'age2', 'sexe', 'date_entry', 'date_discharge', 'date_of_birth',
       'first_name', 'last_name',
       'icd_primary_code', 'icd_primary_description', 'icd_parent_code',
       'icd11_primary_code', 'icd11_primary_description',
       'icd11_primary_code_postcoord_list',       # MODIF CIM-11 : codes post-coordonnés DP tirés (1 par prompt)
       'icd11_primary_libelle_postcoord_list',    # MODIF CIM-11 : libellés complets correspondants
       'icd_secondary_code', 'icd11_secondary_code',
       'icd11_secondary_code_all',  # MODIF : tous les DAS avant filtrage cohérence
       'case_management_type', 'case_management_type_description', 'coding_rule', 'case_management_type_text',
       'drg_parent_code', 'drg_parent_description',
       'text_secondary_icd_official',
       'procedure', 'text_procedure',
       'admission_type', 'admission_mode', 'discharge_disposition', 'los',
       'cancer_stage', 'score_TNM', 'histological_type',
       'treatment_recommandation', 'chemotherapy_regimen', 'biomarkers',
       'first_name_med', 'last_name_med',
       'department', 'hospital', 'template_name',
       ] + user_prompt_cols + ['system_prompt', 'prefix', 'prefix_len']

df_scenario = df_list[keep_cols]
df_scenario.to_csv(run_dir + "generated_scenarios_" + dt.datetime.now().strftime('%Y%m%d_%H%M') + ".csv")
print(f"Scénarios sauvegardés : {len(df_scenario)} lignes")
print(f"Colonnes user_prompt disponibles : {user_prompt_cols}")

Dossier de sortie : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/resultats/20260721_154509/
Scénarios sauvegardés : 200 lignes
Colonnes user_prompt disponibles : ['user_prompt_1', 'user_prompt_2', 'user_prompt_3', 'user_prompt_4', 'user_prompt_5']


In [29]:
print(pd.DataFrame(list_scenario).columns.tolist())

['age', 'sexe', 'date_entry', 'date_discharge', 'date_of_birth', 'first_name', 'last_name', 'icd_primary_code', 'case_management_type', 'icd_secondary_code', 'text_secondary_icd_official', 'procedure', 'icd_primary_description', 'admission_mode', 'discharge_disposition', 'cancer_stage', 'score_TNM', 'histological_type', 'treatment_recommandation', 'chemotherapy_regimen', 'biomarkers', 'department', 'hospital', 'first_name_med', 'last_name_med', 'template_name', 'index', 'admission_type', 'drg_parent_code', 'ghm2', 'nb_associated', 'nb', 'age2', 'los', 'drg_parent_description', 'da', 'libelle_da', 'gp_cas', 'libelle_gp_cas', 'ga', 'libelle_ga', 'da_gp', 'da_gp_ga', 'anseqta', 'aso', 'specialty', 'icd11_primary_code', 'icd11_secondary_code', 'icd_parent_code', 'departement', 'case_management_type_description', 'text_procedure', 'case_management_type_text', 'coding_rule', 'case_management_description', 'icd11_primary_description', 'icd11_secondary_code_all', 'icd11_primary_code_postcoord_

###  Test — Vérification des scénarios générés
Vérifie que les prompts contiennent bien les codes CIM-11 et les codes post-coordonnés.

In [30]:
# Vérification des 3 premiers scénarios
print("=== Vérification des scénarios générés ===")
for i, scenario in enumerate(list_scenario[:3]):
    nb_prompts = sum(1 for k in scenario if k.startswith('user_prompt_'))
    print(f"\nSCÉNARIO {i+1}")
    print(f"  CIM-10 : {scenario['icd_primary_code']} → CIM-11 : {scenario['icd11_primary_code']}")
    print(f"  Libellé CIM-11 : {scenario['icd11_primary_description']}")
    print(f"  Nb prompts générés : {nb_prompts}")
    print(f"  Template : {scenario['template_name']}")
    if 'user_prompt_1' in scenario:
        prompt = scenario['user_prompt_1']
        # Extraire la partie codes du prompt
        debut = prompt.find('Codage CIM-11')
        fin = prompt.find('Mode d\'entrée')
        if debut > 0 and fin > 0:
            print(f"  Extrait prompt :\n{prompt[debut:fin].strip()}")

=== Vérification des scénarios générés ===

SCÉNARIO 1
  CIM-10 : O800 → CIM-11 : JB20.0
  Libellé CIM-11 : Accouchement spontané par présentation du sommet
  Nb prompts générés : 1
  Template : delivery_inpatient_hospit.txt

SCÉNARIO 2
  CIM-10 : O610 → CIM-11 : JB01.0
  Libellé CIM-11 : Échec du déclenchement médical du travail
  Nb prompts générés : 1
  Template : delivery_inpatient_csection_hospit.txt

SCÉNARIO 3
  CIM-10 : O800 → CIM-11 : JB20.0
  Libellé CIM-11 : Accouchement spontané par présentation du sommet
  Nb prompts générés : 1
  Template : delivery_inpatient_hospit.txt


## 18. Transformation du DataFrame pour l'envoi à Mistral
> **MODIF CIM-11** : Cellule ajoutée.
>
> `df_scenario` a une ligne par séjour avec N colonnes `user_prompt_i`.
> L'API Mistral a besoin d'une ligne par prompt.
> On transforme donc en créant autant de lignes que de prompts disponibles par séjour.
>
> Les colonnes `sejour_idx` et `prompt_num` permettront d'identifier chaque CRH dans le fichier final.

In [31]:
# MODIF CIM-11 : Transformation du DataFrame pour l'envoi à Mistral
# On crée autant de lignes que de prompts disponibles par séjour
# La boucle while s'arrête dès qu'une colonne user_prompt_j n'existe pas ou est vide (NaN)
rows_mistral = []
for idx, row in df_scenario.iterrows():
    j = 1
    while f'user_prompt_{j}' in df_scenario.columns and pd.notna(row.get(f'user_prompt_{j}')):
        row_mistral = row.copy()
        row_mistral['user_prompt'] = row[f'user_prompt_{j}']  # Le prompt à envoyer à Mistral
        row_mistral['sejour_idx'] = idx                        # Index du séjour original
        row_mistral['prompt_num'] = j                          # Numéro du prompt (pour le pivot final)

        # MODIF CIM-11 : Sélection du code/libellé post-coordonné DP correspondant à CE prompt précis
        # (le j-ème élément de la liste correspond au j-ème prompt, j commence à 1)
        liste_codes = row.get('icd11_primary_code_postcoord_list')
        liste_libelles = row.get('icd11_primary_libelle_postcoord_list')
        if isinstance(liste_codes, list) and len(liste_codes) >= j:
            row_mistral['icd11_primary_code_postcoord'] = liste_codes[j - 1]
            row_mistral['icd11_primary_libelle_postcoord'] = liste_libelles[j - 1]
        else:
            row_mistral['icd11_primary_code_postcoord'] = row.get('icd11_primary_code')
            row_mistral['icd11_primary_libelle_postcoord'] = row.get('icd11_primary_description')

        rows_mistral.append(row_mistral)
        j += 1

df_mistral = pd.DataFrame(rows_mistral).reset_index(drop=True)
print(f"Prompts à envoyer à Mistral : {len(df_mistral)}")
print(f"(soit en moyenne {len(df_mistral)/N_SCENARIO:.1f} prompts par séjour)")

Prompts à envoyer à Mistral : 446
(soit en moyenne 2.2 prompts par séjour)


## 19. Envoi des prompts à Mistral en batch
> **MODIF CIM-11** : Cellule modifiée.
>
> **`math.ceil` au lieu de `int`** : `int(5/100) = 0` → la boucle ne s'exécuterait jamais pour les petits batches. `math.ceil(5/100) = 1` → la boucle s'exécute une fois.
>
> **`df_mistral` au lieu de `df_scenario`** : on envoie les lignes transformées (une par prompt).

In [32]:
# MODIF CIM-11 : Test - un seul CRH via l'API Batch (pour comparer avec le mode chat.complete)
prompt_unique = df_mistral.iloc[[0]].reset_index(drop=True)  # une seule ligne, sous forme de DataFrame

input_file_test = create_input_file(client, prompt_unique)
print(f"Fichier d'entrée créé : {input_file_test}")

batch_job_test = client.batch.jobs.create(
    input_files=[input_file_test.id],
    model=model,
    endpoint="/v1/chat/completions",
    metadata={"job_type": "test_unitaire"}
)
print(f"Job créé : {batch_job_test.id} | statut initial : {batch_job_test.status}")

Fichier d'entrée créé : id='5cfaaf08-87f4-4149-ae9a-1df881a2b048' object='file' bytes=8793 created_at=1784648710 filename='file.jsonl' purpose='batch' sample_type='batch_request' source='upload' num_lines=1
Job créé : e6bf0bc4-71ca-4c3f-b36c-f3ced75e4a1a | statut initial : QUEUED


In [33]:
statut_test = client.batch.jobs.get(job_id=batch_job_test.id)
print(f"Statut : {statut_test.status} | complétées : {statut_test.completed_requests}/{statut_test.total_requests}")

Statut : RUNNING | complétées : 0/1


In [34]:
for i in range(0, max(1, math.ceil(len(df_mistral)/n_samples_by_batch))):
    print(f"Batch {i} : prompts {i*n_samples_by_batch} - {min((i+1)*n_samples_by_batch, len(df_mistral))}")
    prompts = df_mistral[i*n_samples_by_batch:(i+1)*n_samples_by_batch].reset_index(drop=True)
    input_file = create_input_file(client, prompts)
    print(f"Fichier d'entrée créé : {input_file}")
    batch_job = run_batch_job(client, input_file, model)
    print(f"Durée du job : {batch_job.completed_at - batch_job.created_at} secondes")
    download_file(client, batch_job.output_file, run_dir + "tmp/batch_" + str(i) + ".json")
    prompts.to_csv(run_dir + "tmp/batch_" + str(i) + ".csv")

Batch 0 : prompts 0 - 100
Fichier d'entrée créé : id='10c9cf96-fb06-4626-8e87-b117b7c157bd' object='file' bytes=848451 created_at=1784648712 filename='file.jsonl' purpose='batch' sample_type='batch_request' source='upload' num_lines=100
Batch job 1167f2cc-7776-4220-807c-d4bafb70b360 completed with status: SUCCESS
Durée du job : 333 secondes
Downloaded file to /content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/resultats/20260721_154509/tmp/batch_0.json
Batch 1 : prompts 100 - 200
Fichier d'entrée créé : id='2668d3c0-b2ce-48bb-96ea-7caf8e12bf1b' object='file' bytes=879707 created_at=1784649047 filename='file.jsonl' purpose='batch' sample_type='batch_request' source='upload' num_lines=100
Batch job c3cab5b5-7c4e-4a1d-9fad-a8b1469d85a5 completed with status: SUCCESS
Durée du job : 305 secondes
Downloaded file to /content/drive/MyDrive/Colab_Notebooks/Serenic_M/generation_crh/resultats/20260721_154509/tmp/batch_1.json
Batch 2 : prompts 200 - 300
Fichier d'entrée créé : id='696b

In [35]:
jobs = client.batch.jobs.list()
for job in jobs.data[:10]:
    print(f"ID: {job.id} | statut: {job.status} | créé: {job.created_at} | complétées: {job.completed_requests}/{job.total_requests}")

ID: 927dc0e8-055a-4950-9cd2-bcd829f58785 | statut: SUCCESS | créé: 1784649832 | complétées: 46/46
ID: fa868068-4542-4f1d-910d-f0f287c0fcda | statut: SUCCESS | créé: 1784649604 | complétées: 100/100
ID: 5a92df3c-6ccd-4f60-adb8-e9c4e6f64763 | statut: SUCCESS | créé: 1784649357 | complétées: 100/100
ID: c3cab5b5-7c4e-4a1d-9fad-a8b1469d85a5 | statut: SUCCESS | créé: 1784649048 | complétées: 100/100
ID: 1167f2cc-7776-4220-807c-d4bafb70b360 | statut: SUCCESS | créé: 1784648712 | complétées: 100/100
ID: e6bf0bc4-71ca-4c3f-b36c-f3ced75e4a1a | statut: SUCCESS | créé: 1784648711 | complétées: 1/1
ID: 3b293877-33f9-4018-b356-dd993dc0f151 | statut: SUCCESS | créé: 1784646733 | complétées: 35/35
ID: 7d5e8d1a-6ff9-4619-b6db-172508e231c3 | statut: SUCCESS | créé: 1784646616 | complétées: 100/100
ID: 54c1b913-8cc9-4657-b983-ffb650ffb809 | statut: SUCCESS | créé: 1784646436 | complétées: 100/100
ID: ecc98a3d-9373-4045-833a-62aa0ae97073 | statut: SUCCESS | créé: 1784646220 | complétées: 100/100


In [36]:
# Remplace par l'ID du job SUCCESS concerné et le numéro de batch correspondant
job_id_a_recuperer = "COLLE_L_ID_ICI"
numero_batch = 0  # adapte selon la position dans la boucle

job_termine = client.batch.jobs.get(job_id=job_id_a_recuperer)
if job_termine.status == "SUCCESS":
    download_file(client, job_termine.output_file, run_dir + f"tmp/batch_{numero_batch}.json")
    print(f"Batch {numero_batch} récupéré manuellement.")
else:
    print(f"Job pas encore terminé (statut : {job_termine.status})")

SDKError: API error occurred: Status 404
{"detail": "Not Found"}

In [ ]:
# Annuler le job bloqué
client.batch.jobs.cancel(job_id="52706b01-0e3b-42e1-9327-a9f35a485c3b")

In [ ]:
# Test rapide de l'API Mistral (hors batch)
try:
    response = client.chat.complete(
        model=model,
        messages=[{"role": "user", "content": "Réponds juste 'ok'"}],
        max_tokens=10
    )
    print("API OK, réponse :", response.choices[0].message.content)
except Exception as e:
    print("Erreur API :", e)

###  Test — Vérification de la transformation
Vérifie que `df_mistral` contient bien le bon nombre de lignes.

In [ ]:
# Vérification de la transformation
print("=== Vérification df_mistral ===")
print(f"Lignes dans df_scenario (séjours) : {len(df_scenario)}")
print(f"Lignes dans df_mistral (prompts)  : {len(df_mistral)}")MI
print(f"Moyenne prompts par séjour        : {len(df_mistral)/len(df_scenario):.1f}")

# Vérifier que chaque séjour a bien ses prompts
print("\nNb de prompts par séjour :")
print(df_mistral.groupby('sejour_idx')['prompt_num'].count().value_counts().sort_index())

## 20. Récupération des résultats
> **MODIF CIM-11** : Cellule modifiée.
>
> **Parsing corrigé** : le format retourné par Mistral est `{"id":..., "custom_id":..., "response":..., "error":...}`. On extrait uniquement `content` (le texte du CRH) et `custom_id` (l'identifiant pour la jointure).
>
> **Une ligne par CRH** : pas de pivot — on sauvegarde directement `df_res_final` qui contient une ligne par CRH généré.

In [ ]:
# MODIF CIM-11 : Récupération des CRH — une ligne par CRH (sans pivot)
df_res_final = pd.DataFrame()
output_dir = run_dir + "tmp"

for i in range(max(1, math.ceil(len(df_mistral)/n_samples_by_batch))):
    file_path = os.path.join(output_dir, f"batch_{i}.json")
    if os.path.exists(file_path):
        prep_dict = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line.strip())
                # Extraction du texte brut depuis la structure de réponse Mistral
                content = data['response']['body']['choices'][0]['message']['content']
                if len(content) < 40:  # On ignore les réponses trop courtes (erreurs)
                    continue
                # MODIF CIM-11 : parsing du JSON pour séparer CR et formulations
                try:
                    match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
                    parsed = json.loads(match.group(1)) if match else json.loads(content)
                    crh_text = parsed["CR"]
                    formulations_str = json.dumps(parsed.get("formulations", {}), ensure_ascii=False)
                except Exception:
                    # Si parsing échoue, on conserve le contenu brut dans crh
                    crh_text = content
                    formulations_str = "{}"
                prep_dict.append({
                    "num_in_": data['custom_id'],
                    "crh": crh_text,
                    "formulations": formulations_str
                })

        # Chargement du CSV du batch (contient les scénarios envoyés)
        df_scenarios_save = pd.read_csv(output_dir + "/batch_" + str(i) + ".csv", index_col=0)
        df_res_tmp = pd.DataFrame(prep_dict)
        df_res_tmp["num_in_"] = df_res_tmp["num_in_"].astype(int)
        # Jointure entre les CRH générés et les scénarios via l'identifiant
        df_res_tmp = df_scenarios_save.merge(df_res_tmp, right_on="num_in_", left_index=True)
        df_res_final = pd.concat([df_res_final, df_res_tmp])

print(f"CRH récupérés : {len(df_res_final)} (une ligne par CRH)")


## 21. Sauvegarde du fichier final
Sauvegarde du fichier final `clinical_report_from_generated_scenarios_cim11.csv`.

**Structure du fichier** : **une ligne par CRH** avec :
- Les données du patient (âge, sexe, codes CIM-10 et CIM-11, diagnostics...)
- Le prompt envoyé à Mistral (`user_prompt`)
- Le CRH généré par Mistral (`crh`)
- L'identifiant du séjour (`sejour_idx`) et le numéro du prompt (`prompt_num`) pour traçabilité

In [ ]:
# MODIF CIM-11 : Sauvegarde — une ligne par CRH
# Suppression des colonnes user_prompt_2 à user_prompt_5 du fichier final
# On garde uniquement user_prompt (le prompt envoyé pour ce CRH spécifique) et crh
cols_a_supprimer = ['user_prompt_1', 'user_prompt_2', 'user_prompt_3', 'user_prompt_4', 'user_prompt_5']
df_res_final_clean = df_res_final.drop(columns=[c for c in cols_a_supprimer if c in df_res_final.columns])

df_res_final_clean.to_csv(run_dir + "clinical_report_from_" + scenario_file_name + ".csv")
print(f"Fichier final sauvegardé : {run_dir}clinical_report_from_{scenario_file_name}.csv")
print(f"Nombre de CRH : {len(df_res_final_clean)}")
print(f"Colonnes : {df_res_final_clean.columns.tolist()}")

## 21. Export des CRH en fichiers .docx
> **MODIF CIM-11** : Cellule ajoutée.
>
> Chaque CRH est exporté dans un fichier `.docx` individuel dans un sous-dossier `docx/` créé dans le dossier de run horodaté.
>
> **Nommage** : `CRH_001.docx`, `CRH_002.docx`, etc.
>
> **Contenu** : uniquement le texte du CRH (colonne `crh`), avec mise en forme basique (titres `###` → Heading 2, `**texte**` → gras).

In [ ]:
!pip install python-docx -q

In [ ]:
# MODIF CIM-11 : Export de chaque CRH en fichier .docx individuel
# Un sous-dossier docx/ est créé dans le dossier de run horodaté
from docx import Document
from docx.shared import Pt
import re

# Chargement du référentiel CIM-11 pour les libellés
dict_libelles = dict_cim11_libelles

def get_libelle(code):
    """Retourne 'Libellé (CODE)' ou juste '(CODE)' si libellé introuvable."""
    if not isinstance(code, str) or code.strip() == '':
        return None
    libelle = dict_libelles.get(code.strip(), '')
    if libelle:
        return f"{libelle} ({code.strip()})"
    return f"({code.strip()})"

docx_dir = run_dir + "docx/"
os.makedirs(docx_dir, exist_ok=True)

def crh_to_docx(crh_text, filepath):
    """Convertit le texte markdown d'un CRH en fichier .docx."""
    doc = Document()
    style = doc.styles['Normal']
    style.font.name = 'Arial'
    style.font.size = Pt(11)

    for line in crh_text.split('\n'):
        if line.startswith('### '):
            doc.add_heading(line[4:], level=2)
        elif line.startswith('## '):
            doc.add_heading(line[3:], level=1)
        elif line.strip() == '---':
            p = doc.add_paragraph()
            p.add_run('─' * 40)
        elif line.strip() == '':
            doc.add_paragraph()
        else:
            p = doc.add_paragraph()
            parts = re.split(r'(\*\*[^*]+\*\*)', line)
            for part in parts:
                if part.startswith('**') and part.endswith('**'):
                    run = p.add_run(part[2:-2])
                    run.bold = True
                else:
                    p.add_run(part)

    doc.save(filepath)

# Export de tous les CRH
n_exported = 0
n_errors = 0
for idx, row in df_res_final_clean.iterrows():
    try:
        # MODIF CIM-11 : crh contient uniquement le texte du CR (plus de JSON à parser)
        crh_text_only = row.get('crh', '')

        # MODIF CIM-11 : formulations lues depuis la colonne dédiée
        formulations = json.loads(row.get('formulations', '{}') or '{}')

        # 1. Scénario de départ
        user_prompt = row.get('user_prompt', '')
        scenario_text = "## Scénario de départ\n\n" + str(user_prompt)

        # 2. Codes CIM-11 avec libellés
        # MODIF CIM-11 : on affiche le code post-coordonné réellement utilisé pour ce CRH,
        # avec son libellé complet (racine + extensions), au lieu du code racine
        codes_text = "\n\n---\n\n## Codes CIM-11\n"
        code_dp = row.get('icd11_primary_code_postcoord') or row.get('icd11_primary_code', '')
        libelle_dp = row.get('icd11_primary_libelle_postcoord') or row.get('icd11_primary_description', '')
        if isinstance(code_dp, str) and code_dp.strip():
            libelle_affiche = libelle_dp if isinstance(libelle_dp, str) and libelle_dp.strip() else get_libelle(code_dp)
            codes_text += f"\n**Diagnostic principal :** {libelle_affiche} ({code_dp.strip()})"

        secondary_raw = row.get('icd11_secondary_code', '')
        if isinstance(secondary_raw, str) and secondary_raw.strip():
            codes_text += "\n\n**Diagnostics associés :**"
            for code in secondary_raw.split(','):
                lib = get_libelle(code.strip())
                if lib:
                    codes_text += f"\n- {lib}"

        # 3. Dictionnaire de formulations
        formulations_text = ""
        if formulations:
            formulations_text += "\n\n---\n\n## Dictionnaire de formulations"
            diagnostics = formulations.get("diagnostics", {})
            if diagnostics:
                formulations_text += "\n\n### Diagnostics"
                for code, expressions in diagnostics.items():
                    formulations_text += f"\n\n**{code}**"
                    for expr in expressions:
                        formulations_text += f"\n- {expr}"
            informations = formulations.get("informations", {})
            if informations:
                formulations_text += "\n\n### Informations"
                for champ, expressions in informations.items():
                    formulations_text += f"\n\n**{champ}**"
                    for expr in expressions:
                        formulations_text += f"\n- {expr}"

        # Assemblage final : scénario + codes + CR + formulations
        crh_text = scenario_text + codes_text + "\n\n---\n\n" + crh_text_only + formulations_text

    except Exception:
        crh_text = row.get('crh', '')

    if not isinstance(crh_text, str) or len(crh_text) < 10:
        n_errors += 1
        continue
    filename = docx_dir + f"CRH_{n_exported + 1:03d}.docx"
    try:
        crh_to_docx(crh_text, filename)
        n_exported += 1
    except Exception as e:
        print(f"Erreur CRH index {idx} : {e}")
        n_errors += 1

print(f"Export .docx terminé : {n_exported} fichiers créés dans {docx_dir}")
if n_errors > 0:
    print(f"  {n_errors} CRH ignorés (contenu vide ou erreur)")

### Test — Vérification du fichier final
Vérifie la structure du fichier final : une ligne par CRH avec les bonnes colonnes.

In [ ]:
# Vérification de la structure du fichier final
print(f"Nombre de CRH : {len(df_res_final_clean)}")
print(f"Colonnes présentes : {df_res_final_clean.columns.tolist()}")
print(f"\nExemple d'une ligne :")
print(df_res_final_clean[['icd_primary_code', 'icd11_primary_code', 'user_prompt', 'crh']].head(2).to_string())

### Test — Distribution des prompts par séjour
Vérifie combien de prompts ont été générés pour chaque séjour (dépend du nombre de codes post-coordonnés cohérents disponibles).

In [ ]:
for i, scenario in enumerate(list_scenario):
    nb_prompts = sum(1 for k in scenario if k.startswith('user_prompt_'))
    print(f"Scénario {i+1} : {nb_prompts} prompts | code CIM-11 : {scenario.get('icd11_primary_code')}")

###  Test — Vérification du nombre de prompts envoyés et CRH reçus
Vérifie que tous les prompts ont bien été envoyés et que tous les CRH ont été reçus.

In [ ]:
print(f"Prompts envoyés à Mistral : {len(df_mistral)}")
print(f"CRH récupérés : {len(df_res_final)}")

###  Test — Vérification des colonnes de df_scenario
Vérifie que df_scenario contient bien les 5 colonnes user_prompt nécessaires pour l'envoi à Mistral.

In [ ]:
print(f"Lignes dans df_scenario : {len(df_scenario)}")
print(f"Colonnes user_prompt dans df_scenario : {[c for c in df_scenario.columns if c.startswith('user_prompt')]}")